In [ ]:
import sys
import socket

print(f"Node:{socket.gethostname()}")

In [ ]:
import os

os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["POLARS_MAX_THREADS"] = "8"


import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import polars as pl
import seaborn as sns
from pathlib import Path
import duckdb

from scipy.stats import skew, spearmanr

from photometry.models.hapke import HapkeModel
from photometry.core.types import GeometryBatch
from photometry.fitting.least_sq import LeastSquaresFitter


# Set scientific plotting style
sns.set_theme(style="whitegrid", context="paper", font_scale=1.2)
plt.rcParams["figure.figsize"] = (10, 6)


import os

print(os.getcwd())

In [ ]:
# Set up paths and load phase-curve data for all phases

project_root = Path.cwd().resolve()
if not (project_root / "data").exists() and (project_root.parent / "data").exists():
    project_root = project_root.parent



# parquet_path_survey_ellipsoid = project_root / "data" / "06_silver_layer_ellipsoid" / "survey_ellipsoid.parquet"

parquet_path_survey_dsk256_110825 = project_root / "data" / "silver" /"gaskell_dsk256_110825" /"DR_survey_gaskell_dsk256_110825.parquet"

In [ ]:
files = duckdb.sql(f"""
    SELECT image_id, COUNT(*) AS n
    FROM read_parquet('{parquet_path_survey_dsk256_110825}')
    GROUP BY image_id
""").df()

print(f"Distinct images: {len(files)}")
print(files["image_id"].str[-3:].value_counts())

In [ ]:
duckdb.sql("PRAGMA threads=8;")
duckdb.sql("PRAGMA memory_limit='28GB';")


print("Executing High-Speed DuckDB Audit...\n")

# --- 1. Boundary & Triangle Law Checks ---


print("--- 1. Boundary & Physics Audit ---")


audit_df = duckdb.sql(f"""

SELECT 
    COUNT(*) AS total_pixels,
    SUM(CASE WHEN incidence < 0 OR emission < 0 OR phase < 0 
               OR incidence > 90 OR emission > 90 OR phase > 180 
               OR iof <= 0 OR iof > 1.5 THEN 1 ELSE 0 END) AS boundary_violations,
    SUM(CASE WHEN phase < ABS(incidence - emission) 
               OR phase > (incidence + emission) + 0.1 THEN 1 ELSE 0 END) AS triangle_violations
FROM read_parquet('{parquet_path_survey_dsk256_110825}')

""").df()

display(audit_df)

# --- 2. Global Distribution Statistics ---


print("\n--- 2. Global Photometric Statistics ---")


stats_df = duckdb.sql(f"""

SELECT 
    mission_phase,
    COUNT(*) AS total_pixels,
    ROUND(MIN(iof), 4) AS min_iof,
    ROUND(AVG(iof), 4) AS mean_iof,
    ROUND(MAX(iof), 4) AS max_iof,
    ROUND(MIN(phase), 2) AS min_phase_deg,
    ROUND(MAX(phase), 2) AS max_phase_deg
FROM read_parquet('{parquet_path_survey_dsk256_110825}')
GROUP BY mission_phase
ORDER BY mission_phase

""").df()

display(stats_df)

# --- 3. MLE Histogram Binning for Seaborn/Matplotlib ---

print("\n--- 3. Aggregating Distribution for Plotting ---")
# Instead of pulling 670M rows, we group I/F into bins of 0.005 width

hist_df = duckdb.sql(f"""
SELECT 
    ROUND(iof * 200) / 200 AS iof_bin,
    COUNT(*) AS pixel_count
FROM read_parquet('{parquet_path_survey_dsk256_110825}')
GROUP BY 1
ORDER BY 1
""").df()

print("Plotting results...")
plt.figure(figsize=(12, 6))

# Plot the pre-aggregated data as a bar chart (visually identical to a histogram)
sns.barplot(
    data=hist_df[hist_df['iof_bin'] <= 0.25], # Zoom in on the main distribution
    x='iof_bin', 
    y='pixel_count', 
    color='purple',
    width=1.0,
    alpha=0.3
)

# Clean up the X-axis ticks so they aren't crowded
plt.xticks(rotation=45)
ax = plt.gca()
for ind, label in enumerate(ax.get_xticklabels()):
    if ind % 10 == 0:  # Show every 10th label
        label.set_visible(True)
    else:
        label.set_visible(False)

plt.title("I/F Reflectance Distribution")
plt.xlabel("I/F (Binned to 0.005)")
plt.ylabel("Pixel Count")
plt.tight_layout()
plt.show()

In [ ]:
check = duckdb.sql(f"""
    SELECT
        COUNT(*)                                   AS n_rows,
        COUNT(DISTINCT image_id)                   AS n_images,

        -- Incidence Range
        MIN(incidence)                             AS min_incidence,
        MAX(incidence)                             AS max_incidence,

        -- Emission Range
        MIN(emission)                              AS min_emission,
        MAX(emission)                              AS max_emission,

        -- Phase Range
        MIN(phase)                                 AS min_phase,
        MAX(phase)                                 AS max_phase,

        -- Reflectance (I/F) Range
        MIN(iof)                                   AS min_iof,
        MAX(iof)                                   AS max_iof,

        -- Sanity Check
       -- SUM(CASE WHEN image_id NOT LIKE '%F1B%' THEN 1 ELSE 0 END) AS non_F1B_rows

    FROM read_parquet('{parquet_path_survey_dsk256_110825}')

""").df()

display(check)

In [ ]:
stats_df = duckdb.sql(f"""
SELECT 
    AVG(iof) AS iof_mean,
    QUANTILE_CONT(iof, 0.5) AS iof_median,
    SKEWNESS(iof) AS iof_skew
FROM read_parquet('{parquet_path_survey_dsk256_110825}')
""").df()

iof_mean_dsk = stats_df['iof_mean'][0]
iof_median_dsk = stats_df['iof_median'][0]
iof_skew_dsk = stats_df['iof_skew'][0]

print(f"I/F Mean:   {iof_mean_dsk:.4f}")
print(f"I/F Median: {iof_median_dsk:.4f}")
print(f"I/F Skew:   {iof_skew_dsk:.4f} (Positive = Right-tailed)")
print("-" * 40)

# 2. Pre-aggregate the Histogram Bins (Width = 0.005)
hist_df = duckdb.sql(f"""
SELECT 
    ROUND(iof * 200) / 200 AS iof_bin,
    COUNT(*) AS count
FROM read_parquet('{parquet_path_survey_dsk256_110825}')
GROUP BY 1
ORDER BY 1
""").df()

# 3. Plotting the Aggregated Data (Zero RAM Impact)
plt.figure(figsize=(10, 5))

# Filter extreme long-tail outliers just for a cleaner plot (e.g., I/F <= 0.3)
plot_df = hist_df[hist_df['iof_bin'] <= 0.3]

# A bar plot of pre-binned data looks identical to sns.histplot
plt.bar(plot_df['iof_bin'], plot_df['count'], width=0.005, color="purple", alpha=0.3)

plt.axvline(iof_mean_dsk, color="red", linestyle="--", label=f"Mean ({iof_mean_dsk:.3f})")
plt.axvline(iof_median_dsk, color="green", linestyle="-", label=f"Median ({iof_median_dsk:.3f})")

plt.title("Raw Photometric Distribution (I/F)")
plt.xlabel("I/F (Reflectance)")
plt.ylabel("Pixel Count")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Maximize the 32GB SLURM node
duckdb.sql("PRAGMA threads=8;")
duckdb.sql("PRAGMA memory_limit='28GB';")


# Paths

golden_path = Path("data/golden/survey_binned_dsk256_110825_range50.parquet")
golden_path.parent.mkdir(parents=True, exist_ok=True)


binned_dsk256_lt50_query = f"""

SELECT 
    -- 1. Banker's Rounding with Float-Safe Epsilon (< 1e-9)

    (CASE 
        WHEN ABS((phase/5.0) - FLOOR(phase/5.0) - 0.5) < 1e-9
        THEN (CASE WHEN CAST(FLOOR(phase/5.0) AS BIGINT) % 2 = 0 
              THEN FLOOR(phase/5.0) ELSE CEIL(phase/5.0) END)
        ELSE ROUND(phase/5.0) 
    END) * 5.0 AS alpha_grid,

    (CASE 
        WHEN ABS((incidence/5.0) - FLOOR(incidence/5.0) - 0.5) < 1e-9
        THEN (CASE WHEN CAST(FLOOR(incidence/5.0) AS BIGINT) % 2 = 0 
              THEN FLOOR(incidence/5.0) ELSE CEIL(incidence/5.0) END)
        ELSE ROUND(incidence/5.0) 
    END) * 5.0 AS i_grid,

    (CASE 
        WHEN ABS((emission/5.0) - FLOOR(emission/5.0) - 0.5) < 1e-9
        THEN (CASE WHEN CAST(FLOOR(emission/5.0) AS BIGINT) % 2 = 0 
              THEN FLOOR(emission/5.0) ELSE CEIL(emission/5.0) END)
        ELSE ROUND(emission/5.0) 
    END) * 5.0 AS e_grid,

    -- 2. Target Aggregations

    AVG(incidence) AS mean_incidence,
    AVG(emission) AS mean_emission,
    AVG(phase) AS mean_phase,
    AVG(iof) AS mean_iof,
    STDDEV_SAMP(iof) AS std_iof,
    COUNT(*) AS n_pixels

FROM read_parquet('{parquet_path_survey_dsk256_110825}')

-- 3. The Physical Boundary Restored (Triangle Law)

WHERE incidence < 50.0
  AND emission < 50.0
  -- AND iof > 0.01
  AND image_id LIKE '%F1B%'
  AND phase >= ABS(incidence - emission)       
  AND phase <= (incidence + emission) + 0.1

GROUP BY 1, 2, 3

-- 4. Statistical Validity Check

HAVING COUNT(*) >= 10 
   AND STDDEV_SAMP(iof) IS NOT NULL

ORDER BY 1, 2, 3

"""


# Execute the query and pull the tiny result table into Python RAM

df_binned_dsk256_lt50 = duckdb.sql(binned_dsk256_lt50_query).df()

print(f"DuckDB binning complete. Retained {len(df_binned_dsk256_lt50):,} safe geometric cubes.")

# Save the GL so you never have to do this again

df_binned_dsk256_lt50.to_parquet(golden_path, index=False)
print(f"Saved GL to: {golden_path}")
display(df_binned_dsk256_lt50)

In [ ]:
# Maximize the 32GB SLURM node
duckdb.sql("PRAGMA threads=8;")
duckdb.sql("PRAGMA memory_limit='28GB';")


# Paths

golden_path = Path("data/golden/survey_binned_dsk256_110825_range80.parquet")
golden_path.parent.mkdir(parents=True, exist_ok=True)


binned_dsk256_lt80_query = f"""

SELECT 
    -- 1. Banker's Rounding with Float-Safe Epsilon (< 1e-9)

    (CASE 
        WHEN ABS((phase/5.0) - FLOOR(phase/5.0) - 0.5) < 1e-9
        THEN (CASE WHEN CAST(FLOOR(phase/5.0) AS BIGINT) % 2 = 0 
              THEN FLOOR(phase/5.0) ELSE CEIL(phase/5.0) END)
        ELSE ROUND(phase/5.0) 
    END) * 5.0 AS alpha_grid,

    (CASE 
        WHEN ABS((incidence/5.0) - FLOOR(incidence/5.0) - 0.5) < 1e-9
        THEN (CASE WHEN CAST(FLOOR(incidence/5.0) AS BIGINT) % 2 = 0 
              THEN FLOOR(incidence/5.0) ELSE CEIL(incidence/5.0) END)
        ELSE ROUND(incidence/5.0) 
    END) * 5.0 AS i_grid,

    (CASE 
        WHEN ABS((emission/5.0) - FLOOR(emission/5.0) - 0.5) < 1e-9
        THEN (CASE WHEN CAST(FLOOR(emission/5.0) AS BIGINT) % 2 = 0 
              THEN FLOOR(emission/5.0) ELSE CEIL(emission/5.0) END)
        ELSE ROUND(emission/5.0) 
    END) * 5.0 AS e_grid,

    -- 2. Target Aggregations

    AVG(incidence) AS mean_incidence,
    AVG(emission) AS mean_emission,
    AVG(phase) AS mean_phase,
    AVG(iof) AS mean_iof,
    STDDEV_SAMP(iof) AS std_iof,
    COUNT(*) AS n_pixels

FROM read_parquet('{parquet_path_survey_dsk256_110825}')

-- 3. The Physical Boundary Restored (Triangle Law)

WHERE incidence < 80.0
  AND emission < 80.0
  -- AND iof > 0.01
  AND image_id LIKE '%F1B%'
  AND phase >= ABS(incidence - emission)       
  AND phase <= (incidence + emission) + 0.1

GROUP BY 1, 2, 3

-- 4. Statistical Validity Check

HAVING COUNT(*) >= 10 
   AND STDDEV_SAMP(iof) IS NOT NULL

ORDER BY 1, 2, 3

"""


# Execute the query and pull the tiny result table into Python RAM

df_binned_dsk256_lt80 = duckdb.sql(binned_dsk256_lt80_query).df()

print(f" {len(df_binned_dsk256_lt80):,} safe geometric cubes.")

# Save the GL so you never have to do this again

df_binned_dsk256_lt80.to_parquet(golden_path, index=False)
print(f"Saved GL to: {golden_path}")
display(df_binned_dsk256_lt80.head())

In [ ]:
dsk256_lt50_sample = df_binned_dsk256_lt50#.collect().sample(fraction=0.05, seed=42)
plt.scatter(dsk256_lt50_sample["mean_phase"], dsk256_lt50_sample["mean_iof"], alpha=0.8, s=1)
plt.title(f"Real Phase Curve of {len(dsk256_lt50_sample):,} Cubes region: incidence < 50°, emission < 50°")
plt.xlabel("Phase Angle (degrees)")
plt.ylabel("I/F")
plt.grid(True)
plt.show()


dsk256_lt80_sample = df_binned_dsk256_lt80#.collect().sample(fraction=0.05, seed=42)
plt.scatter(dsk256_lt80_sample["mean_phase"], dsk256_lt80_sample["mean_iof"], alpha=0.8, s=1)
plt.title(f"Real Phase Curve of {len(dsk256_lt80_sample):,} Cubes region: incidence < 80°, emission < 80°")
plt.xlabel("Phase Angle (degrees)")
plt.ylabel("I/F")
plt.grid(True)
plt.show()

In [ ]:
## Geometry Ingestion and weights 


valid_mask_dsk256_lt50 = (
    (df_binned_dsk256_lt50["mean_phase"] <= 98.0) & 
    (df_binned_dsk256_lt50  ["std_iof"] > 0.0)    
)


# Re-apply mask correctly on rows (pandas .filter() was dropping columns)
df_binned_cleaned_dsk256_lt50 = df_binned_dsk256_lt50[valid_mask_dsk256_lt50].copy()    

print(f"Domain mask applied. Final cube count: {len(df_binned_cleaned_dsk256_lt50)}")

required_cols = ["mean_incidence", "mean_emission", "mean_phase"]
missing = [c for c in required_cols if c not in df_binned_cleaned_dsk256_lt50.columns]
if missing:
    raise KeyError(f"Missing required columns after masking: {missing}")

geom_clean_dsk256_lt50 = GeometryBatch(
    incidence=np.deg2rad(df_binned_cleaned_dsk256_lt50["mean_incidence"].to_numpy()),
    emission=np.deg2rad(df_binned_cleaned_dsk256_lt50["mean_emission"].to_numpy()),
    phase=np.deg2rad(df_binned_cleaned_dsk256_lt50["mean_phase"].to_numpy())
)




# --- Principled weighting: standard error + systematic floor in quadrature ---

n_pixels_arr_dsk256_lt50 = df_binned_cleaned_dsk256_lt50["n_pixels"].to_numpy()
std_iof_arr_dsk256_lt50  = df_binned_cleaned_dsk256_lt50["std_iof"].to_numpy()
measured_iof_dsk256_lt50 = df_binned_cleaned_dsk256_lt50["mean_iof"].to_numpy()

stat_error_dsk256_lt50 = np.maximum(std_iof_arr_dsk256_lt50, 1e-9) / np.sqrt(n_pixels_arr_dsk256_lt50)

# Derive the floor from YOUR measured residual decomposition, don't assert it:
# CV_other ≈ 8.3% of global mean I/F is the irreducible per-bin systematic
sys_floor_dsk256_lt50 = 0.083 * measured_iof_dsk256_lt50.mean()   # ≈ 0.012 in I/F, derived not assumed

total_error_dsk256_lt50 = np.sqrt(stat_error_dsk256_lt50**2 + sys_floor_dsk256_lt50**2)
weights_dsk256_lt50 = 1.0 / total_error_dsk256_lt50






#std_iof_arr = df_binned_cleaned_dsk["std_iof"].to_numpy()
#weights_inv_sigma_dsk = 1.0 / np.maximum(std_iof_arr, 1e-9)



# --- DIAGNOSTIC BLOCK ---
print(f"Max Phase Angle: {np.rad2deg(geom_clean_dsk256_lt50.phase).max():.2f} deg")

print(f"Max Incidence Angle: {np.rad2deg(geom_clean_dsk256_lt50.incidence).max():.2f} deg")
print(f"Weight Ratio (Max/Min): {weights_dsk256_lt50.max() / weights_dsk256_lt50.min():.2f}")
print(f"Data points passed to fitter: {len(measured_iof_dsk256_lt50)}")



# Check the number of bins
print(f"Total bins in dataset: {len(measured_iof_dsk256_lt50)}")
# Check the average number of pixels per bin
print(f"Average pixels per bin: {df_binned_cleaned_dsk256_lt50['n_pixels'].mean():.2f}")

In [ ]:
## Geometry Ingestion and weights 


valid_mask_dsk256_lt80 = (
    (df_binned_dsk256_lt80["mean_phase"] <= 98.0) & 
    (df_binned_dsk256_lt80  ["std_iof"] > 0.0)    
)


# Re-apply mask correctly on rows (pandas .filter() was dropping columns)
df_binned_cleaned_dsk256_lt80 = df_binned_dsk256_lt80[valid_mask_dsk256_lt80].copy()    

print(f"Domain mask applied. Final cube count: {len(df_binned_cleaned_dsk256_lt80)}")

required_cols = ["mean_incidence", "mean_emission", "mean_phase"]
missing = [c for c in required_cols if c not in df_binned_cleaned_dsk256_lt80.columns]
if missing:
    raise KeyError(f"Missing required columns after masking: {missing}")

geom_clean_dsk256_lt80 = GeometryBatch(
    incidence=np.deg2rad(df_binned_cleaned_dsk256_lt80["mean_incidence"].to_numpy()),
    emission=np.deg2rad(df_binned_cleaned_dsk256_lt80["mean_emission"].to_numpy()),
    phase=np.deg2rad(df_binned_cleaned_dsk256_lt80["mean_phase"].to_numpy())
)




# --- Principled weighting: standard error + systematic floor in quadrature ---

n_pixels_arr_dsk256_lt80 = df_binned_cleaned_dsk256_lt80["n_pixels"].to_numpy()
std_iof_arr_dsk256_lt80 = df_binned_cleaned_dsk256_lt80["std_iof"].to_numpy()
measured_iof_dsk256_lt80 = df_binned_cleaned_dsk256_lt80["mean_iof"].to_numpy()

stat_error_dsk256_lt80 = np.maximum(std_iof_arr_dsk256_lt80, 1e-9) / np.sqrt(n_pixels_arr_dsk256_lt80)

# Derive the floor from YOUR measured residual decomposition, don't assert it:
# CV_other ≈ 8.3% of global mean I/F is the irreducible per-bin systematic
sys_floor_dsk256_lt80 = 0.083 * measured_iof_dsk256_lt80.mean()   # ≈ 0.012 in I/F, derived not assumed

total_error_dsk256_lt80 = np.sqrt(stat_error_dsk256_lt80**2 + sys_floor_dsk256_lt80**2)
weights_dsk256_lt80 = 1.0 / total_error_dsk256_lt80     






#std_iof_arr = df_binned_cleaned_dsk["std_iof"].to_numpy()
#weights_inv_sigma_dsk = 1.0 / np.maximum(std_iof_arr, 1e-9)



# --- DIAGNOSTIC BLOCK ---
print(f"Max Phase Angle: {np.rad2deg(geom_clean_dsk256_lt80.phase).max():.2f} deg")

print(f"Max Incidence Angle: {np.rad2deg(geom_clean_dsk256_lt80.incidence).max():.2f} deg")
print(f"Weight Ratio (Max/Min): {weights_dsk256_lt80.max() / weights_dsk256_lt80.min():.2f}")
print(f"Data points passed to fitter: {len(measured_iof_dsk256_lt80)}")
# ------------------------


# Check the number of bins
print(f"Total bins in dataset: {len(measured_iof_dsk256_lt80)}")
# Check the average number of pixels per bin
print(f"Average pixels per bin: {df_binned_cleaned_dsk256_lt80['n_pixels'].mean():.2f}")

In [ ]:
# Model initiation

model_case1_dsk256_lt50 = HapkeModel(enable_shoe=True, enable_roughness=True, fixed_parameters={'B0': 1.03, 'h': 0.04}) #, 'w': 0.511, 'g': -0.294})


fitter = LeastSquaresFitter()


results_summary_dsk256_lt50 = {}



print(f"Optimizing: {model_case1_dsk256_lt50.parameter_names()}")

# Verify the hijack worked
#print(f"\nModel initialized. Active parameters mapped to Jacobian: {model_case1.parameter_names()}")
#print(f"Fixed parameters running in background: B0={model_case1.parameters['B0']}, h={model_case1.parameters['h']}, theta_bar={model_case1.parameters['theta_bar']} degrees") # theta_bar fixed
#print(f"Fixed parameters running in background: B0={model_case1.parameters['B0']}, h={model_case1.parameters['h']}")
#print(f"Fixed parameters running in background: theta_bar={model.parameters['theta_bar']} degrees") # theta_bar fixed





# Multi-Start Generator (5D Space)

n_starts = 100
np.random.seed(42)
initial_guesses_case1_dsk256_lt50 = [
    {
        "w": np.random.uniform(0.3, 0.6),
        "g": np.random.uniform(-0.4, -0.2),
        "theta_bar":np.random.uniform(1.0,30.0),#(1.0, 30.0), # in degrees, will be converted to radians inside the model
       #'B0': np.random.uniform(0.3, 2.0),  # SHOE Amplitude
      # 'h': np.random.uniform(0.01, 0.15)   #
    }

    for _ in range(n_starts)
]

best_cost_case1_dsk256_lt50 = np.inf
best_result_case1_dsk256_lt50 = None

for guess in initial_guesses_case1_dsk256_lt50:
    model_case1_dsk256_lt50.parameters.update(guess)

    result = fitter.fit(
        model=model_case1_dsk256_lt50,
        geometry=geom_clean_dsk256_lt50,
        observed_reflectance=measured_iof_dsk256_lt50,
        weights=weights_dsk256_lt50
    )

    if result.metadata["success"] and result.objective_value < best_cost_case1_dsk256_lt50:
        best_cost_case1_dsk256_lt50 = result.objective_value
        best_result_case1_dsk256_lt50 = result

if best_result_case1_dsk256_lt50 is None:
    raise RuntimeError("Optimization collapsed. No multi-start vectors converged.")

#  

print("Parameter fitting complete.")
print(f"Cost: {best_result_case1_dsk256_lt50.objective_value:.4f}")

for param_name, value in best_result_case1_dsk256_lt50.fitted_parameters.items():
    print(f"{param_name}: {value:.4f}")

print("Parameter uncertainties (1-Sigma)")
if best_result_case1_dsk256_lt50.metadata.get("parameter_errors"):
    for param_name, error in best_result_case1_dsk256_lt50.metadata["parameter_errors"].items():
        print(f"{param_name}: +/- {error:.4f}")
else:
    print("Jacobian failed to invert (Flat gradient detected).")


results_summary_dsk256_lt50["Case 1 (3-Param)"] = best_result_case1_dsk256_lt50.fitted_parameters.copy()

In [ ]:
best_result_case1_dsk256_lt50.fitted_parameters

In [ ]:
# ==========================================
# 0. Model Evaluation
# ==========================================


model_eval_dsk256_lt50 = HapkeModel(
    enable_shoe=True,
    enable_roughness=True,
    fixed_parameters={"B0": 1.03, "h": 0.04},
)

model_eval_dsk256_lt50.parameters.update(best_result_case1_dsk256_lt50.fitted_parameters)


predicted_iof_dsk256_lt50 = model_eval_dsk256_lt50._reflectance_numpy(
    geom_clean_dsk256_lt50
)
residuals_dsk256_lt50 = (
    df_binned_cleaned_dsk256_lt50["mean_iof"].to_numpy() - predicted_iof_dsk256_lt50        
)

# ==========================================
# 1. Synthetic Pure Phase Curve
#  
# ==========================================
alpha_smooth = np.linspace(
    df_binned_cleaned_dsk256_lt50["mean_phase"].min(),
    df_binned_cleaned_dsk256_lt50["mean_phase"].max(),
    200
)
geom_fixed = GeometryBatch(
    incidence=np.full_like(alpha_smooth, np.deg2rad(30.0)),
    emission=np.full_like(alpha_smooth, np.deg2rad(0.0)),
    phase=np.deg2rad(alpha_smooth)
)
predicted_iof_fixed = model_eval_dsk256_lt50._reflectance_numpy(geom_fixed)

# ==========================================
# 2. Two-Panel Diagnostic Plot
# ==========================================
fig, (ax_main, ax_res) = plt.subplots(
    nrows=2, ncols=1,
    figsize=(10, 8),
    dpi=150,
    sharex=True,
    gridspec_kw={'height_ratios': [3, 1], 'hspace': 0.05}
)

# --- Top panel ---
ax_main.axvspan(80, 98, color='gray', alpha=0.15, zorder=0)
ax_main.text(
    89, 0.22,
    "Model Divergence Zone\n(Phase > 80°)",
    fontsize=10, ha='center', va='center', color='darkred',
    fontweight='bold',
    bbox=dict(facecolor='white', alpha=0.8, edgecolor='none', pad=3)
)

ax_main.scatter(
    df_binned_cleaned_dsk256_lt50["mean_phase"],
    predicted_iof_dsk256_lt50,
    facecolors='none', edgecolors='black',
    alpha=0.3, s=10, zorder=1,
    label='Hapke Model Prediction'
)

scatter = ax_main.scatter(
    df_binned_cleaned_dsk256_lt50["mean_phase"],
    df_binned_cleaned_dsk256_lt50["mean_iof"],
    c=df_binned_cleaned_dsk256_lt50["mean_emission"],
    cmap='viridis', alpha=0.8, s=10, zorder=2,
    label='Real Data'
)

ax_main.plot(
    alpha_smooth, predicted_iof_fixed,
    color='black', linewidth=2.5, linestyle='--', zorder=3,
    label='Pure Phase Curve (i=30°, e=0°)'
)

ax_main.set_title(
    f"Diagnostic Phase Curve and Residuals "
    f"({len(df_binned_cleaned_dsk256_lt50):,} Cubes)\n"
    f"Region: Incidence < 50°, Emission < 50°",
    fontweight='bold'
)
ax_main.set_ylabel("I/F")
ax_main.grid(True, linestyle='--', alpha=0.5)
ax_main.legend(loc='upper right', framealpha=0.9)

cbar = fig.colorbar(scatter, ax=ax_main, aspect=20)
cbar.set_label('Mean Emission Angle (degrees)', rotation=270, labelpad=15)

# --- Bottom panel ---
ax_res.axhline(0, color='black', linewidth=1.5, linestyle='--', zorder=1)
ax_res.axvspan(80, 98, color='gray', alpha=0.15, zorder=0)

ax_res.scatter(
    df_binned_cleaned_dsk256_lt50["mean_phase"],
    residuals_dsk256_lt50,
    c=df_binned_cleaned_dsk256_lt50["mean_emission"],
    cmap='viridis', alpha=0.8, s=10, zorder=2
)

ax_res.text(
    0.02, 0.85,
    '+ = asteroid brighter than model',
    transform=ax_res.transAxes, fontsize=8, color='gray'
)

res_max = max(abs(residuals_dsk256_lt50.min()), abs(residuals_dsk256_lt50.max())) * 1.1
ax_res.set_ylim(-res_max, res_max)
ax_res.set_xlabel("Phase Angle (degrees)")
ax_res.set_ylabel("Residual (Data − Model)")
ax_res.grid(True, linestyle='--', alpha=0.5)

plt.show()

In [ ]:
obs   = df_binned_cleaned_dsk256_lt50["mean_iof"].to_numpy()
pred  = predicted_iof_dsk256_lt50
phase = df_binned_cleaned_dsk256_lt50["mean_phase"].to_numpy()

# correction factor (same as used for the images)

geom_std = GeometryBatch(
    incidence=np.deg2rad(np.full(len(df_binned_cleaned_dsk256_lt50), 30.0)),
    emission=np.deg2rad(np.zeros(len(df_binned_cleaned_dsk256_lt50))),
    phase=np.deg2rad(np.full(len(df_binned_cleaned_dsk256_lt50), 30.0))
)
pred_std    = model_eval_dsk256_lt50._reflectance_numpy(geom_std)
corr_factor = np.clip(pred_std / np.maximum(pred, 1e-6), 0.5, 1.8)
iof_corr    = obs * corr_factor

# ── bin by 5-degree phase intervals, restrict to 10–80° ──────────────────────
mask        = (phase >= 10) & (phase <= 80)
phase_m     = phase[mask]
raw_m       = obs[mask]
corr_m      = iof_corr[mask]

bins        = np.arange(10, 85, 5)
bin_centres = (bins[:-1] + bins[1:]) / 2
raw_binned  = [raw_m[(phase_m >= b) & (phase_m < b+5)].mean()
               for b in bins[:-1]]
corr_binned = [corr_m[(phase_m >= b) & (phase_m < b+5)].mean()
               for b in bins[:-1]]

# ── slopes ────────────────────────────────────────────────────────────────────
raw_slope  = np.polyfit(bin_centres, raw_binned,  1)[0]
corr_slope = np.polyfit(bin_centres, corr_binned, 1)[0]
reduction  = (1 - abs(corr_slope / raw_slope)) * 100
print(f"Raw slope  : {raw_slope:+.5f} I/F/deg")
print(f"Corr slope : {corr_slope:+.5f} I/F/deg")
print(f"Reduction  : {reduction:.1f}%")

# ── plot ──────────────────────────────────────────────────────────────────────
BG, fg = '#0d0d0d', '#e0e0e0'
fig, ax = plt.subplots(figsize=(10, 5), dpi=200, facecolor=BG)
ax.set_facecolor(BG)

ax.plot(bin_centres, raw_binned,
        color='#a0a0a0', lw=2, marker='o', ms=5,
        label='Raw I/F  (geometry-dependent)')
ax.plot(bin_centres, corr_binned,
        color='#68d391', lw=2.5, marker='s', ms=5,
        label='Corrected I/F  (surface albedo)')

# ── 96.2% annotation ─────────────────────────────────────────────────────────
# find midpoint of the gap around α=45° for the arrow
mid_idx  = np.argmin(np.abs(bin_centres - 45))
y_raw_45 = raw_binned[mid_idx]
y_cor_45 = corr_binned[mid_idx]
y_mid    = (y_raw_45 + y_cor_45) / 2

ax.annotate(
    f"{reduction:.1f}% slope reduction",
    xy=(45, y_cor_45 + 0.003),
    xytext=(52, y_mid + 0.025),
    fontsize=11, fontweight='bold', color='#68d391',
    arrowprops=dict(arrowstyle='->', color='#68d391', lw=1.5),
    ha='left'
)
# bracket lines showing the gap
ax.annotate('', xy=(44, y_cor_45), xytext=(44, y_raw_45),
            arrowprops=dict(arrowstyle='<->', color='#68d391',
                            lw=1.2, mutation_scale=12))

# ── styling ───────────────────────────────────────────────────────────────────
ax.set_xlabel('Phase angle  α  (degrees)', color=fg, fontsize=12)
ax.set_ylabel('Mean I/F', color=fg, fontsize=12)
ax.set_title('Phase-curve flattening after photometric correction\n'
             'Survey F1B · i<50°, e<50° · 1,010 bins',
             color=fg, fontsize=12, pad=10)

ax.tick_params(colors=fg, labelsize=10)
for spine in ax.spines.values():
    spine.set_color('#444')
ax.grid(True, linestyle='--', alpha=0.25, color='#666')

legend = ax.legend(fontsize=10, framealpha=0.15,
                   facecolor='#1a1a1a', edgecolor='#444',
                   labelcolor=fg)

plt.tight_layout(pad=1.2)
plt.savefig('results/phase_curve_flattening_poster.png',
            dpi=200, bbox_inches='tight',
            facecolor=BG, pad_inches=0.15)
plt.show()

In [ ]:
# Model initiation

model_case1_dsk256_lt80 = HapkeModel(enable_shoe=True, enable_roughness=True, fixed_parameters={'B0': 1.03, 'h': 0.04}) #, 'w': 0.511, 'g': -0.294})


fitter = LeastSquaresFitter()


results_summary_dsk256_lt80 = {}



print(f"Optimizing: {model_case1_dsk256_lt80.parameter_names()}")

# Verify the hijack worked
#print(f"\nModel initialized. Active parameters mapped to Jacobian: {model_case1.parameter_names()}")
#print(f"Fixed parameters running in background: B0={model_case1.parameters['B0']}, h={model_case1.parameters['h']}, theta_bar={model_case1.parameters['theta_bar']} degrees") # theta_bar fixed
#print(f"Fixed parameters running in background: B0={model_case1.parameters['B0']}, h={model_case1.parameters['h']}")
#print(f"Fixed parameters running in background: theta_bar={model.parameters['theta_bar']} degrees") # theta_bar fixed





# Multi-Start Generator (5D Space)

n_starts = 100
np.random.seed(42)
initial_guesses_case1_dsk256_lt80 = [
    {
        "w": np.random.uniform(0.3, 0.6),
        "g": np.random.uniform(-0.4, -0.2),
        "theta_bar":np.random.uniform(1.0,30.0),#(1.0, 30.0), # in degrees, will be converted to radians inside the model
       #'B0': np.random.uniform(0.3, 2.0),  # SHOE Amplitude
      # 'h': np.random.uniform(0.01, 0.15)   #
    }

    for _ in range(n_starts)
]

best_cost_case1_dsk256_lt80 = np.inf
best_result_case1_dsk256_lt80 = None

for guess in initial_guesses_case1_dsk256_lt80:
    model_case1_dsk256_lt80.parameters.update(guess)

    result = fitter.fit(
        model=model_case1_dsk256_lt80,
        geometry=geom_clean_dsk256_lt80,
        observed_reflectance=measured_iof_dsk256_lt80,
        weights=weights_dsk256_lt80
    )

    if result.metadata["success"] and result.objective_value < best_cost_case1_dsk256_lt80:
        best_cost_case1_dsk256_lt80 = result.objective_value
        best_result_case1_dsk256_lt80 = result

if best_result_case1_dsk256_lt80 is None:
    raise RuntimeError("Optimization collapsed. No multi-start vectors converged.")

#  

print("Parameter fitting complete.")
print(f"Cost: {best_result_case1_dsk256_lt80.objective_value:.4f}")

for param_name, value in best_result_case1_dsk256_lt80.fitted_parameters.items():
    print(f"{param_name}: {value:.4f}")

print("Parameter uncertainties (1-Sigma)")
if best_result_case1_dsk256_lt80.metadata.get("parameter_errors"):
    for param_name, error in best_result_case1_dsk256_lt80.metadata["parameter_errors"].items():
        print(f"{param_name}: +/- {error:.4f}")
else:
    print("Jacobian failed to invert (Flat gradient detected).")


results_summary_dsk256_lt80["Case 1 (3-Param)"] = best_result_case1_dsk256_lt80.fitted_parameters.copy()

In [ ]:
# Model initiation

model_case2_dsk256_lt50 = HapkeModel(enable_shoe=True, enable_roughness=True, fixed_parameters={'B0': 1.03})





fitter = LeastSquaresFitter()

results_summary_dsk256_lt50 = {}


print(f"Optimizing: {model_case2_dsk256_lt50.parameter_names()}")







# Multi-Start Generator (5D Space)

n_starts = 100
np.random.seed(42)
initial_guesses_case2_dsk256_lt50 = [
    {
        "w": np.random.uniform(0.3, 0.6),
        "g": np.random.uniform(-0.4, -0.2),
        "theta_bar":np.random.uniform(1.0,30.0),#(1.0, 30.0), # in degrees, will be converted to radians inside the model
        'h': np.random.uniform(0.01, 0.15)   #
    }

    for _ in range(n_starts)
]

best_cost_case2_dsk256_lt50 = np.inf
best_result_case2_dsk256_lt50 = None

for guess in initial_guesses_case2_dsk256_lt50:
    model_case2_dsk256_lt50.parameters.update(guess)

    result = fitter.fit(
        model=model_case2_dsk256_lt50,
        geometry=geom_clean_dsk256_lt50,
        observed_reflectance=measured_iof_dsk256_lt50,
        weights=weights_dsk256_lt50
    )

    if result.metadata["success"] and result.objective_value < best_cost_case2_dsk256_lt50:
        best_cost_case2_dsk256_lt50 = result.objective_value
        best_result_case2_dsk256_lt50 = result

if best_result_case2_dsk256_lt50 is None:
    raise RuntimeError("Optimization collapsed. No multi-start vectors converged.")

#  

print("Parameter fitting complete.")
print(f"Cost: {best_result_case2_dsk256_lt50.objective_value:.4f}")

for param_name, value in best_result_case2_dsk256_lt50.fitted_parameters.items():
    print(f"{param_name}: {value:.4f}")

print("Parameter uncertainties (1-Sigma)")
if best_result_case2_dsk256_lt50.metadata.get("parameter_errors"):
    for param_name, error in best_result_case2_dsk256_lt50.metadata["parameter_errors"].items():
        print(f"{param_name}: +/- {error:.4f}")
else:
    print("Jacobian failed to invert (Flat gradient detected).")


results_summary_dsk256_lt50["Case 2 (4-Param)"] = best_result_case2_dsk256_lt50.fitted_parameters.copy()

In [ ]:
if "Case 1 (3-Param)" in results_summary_dsk256_lt50:
    best_result_dsk256_lt50 = results_summary_dsk256_lt50["Case 1 (3-Param)"]
else:
    best_result_dsk256_lt50 = best_result_case1_dsk256_lt50.fitted_parameters

model_eval_dsk256_lt50 = HapkeModel(enable_shoe=True, enable_roughness=True, fixed_parameters={'B0': 1.03, 'h': 0.04})
model_eval_dsk256_lt50.parameters.update(best_result_dsk256_lt50)


# 2. Generate Predictions
predicted_iof_dsk256_lt50 = model_eval_dsk256_lt50._reflectance_numpy(geom_clean_dsk256_lt50)

# 3. Setup Arrays for Math
obs_dsk256_lt50  = measured_iof_dsk256_lt50
pred_dsk256_lt50 = predicted_iof_dsk256_lt50
resid_dsk256_lt50 = obs_dsk256_lt50 - pred_dsk256_lt50
n_bins_dsk256_lt50 = len(obs_dsk256_lt50)
n_params_dsk256_lt50 = len(best_result_case1_dsk256_lt50.fitted_parameters)
dof_dsk256_lt50 = n_bins_dsk256_lt50 - n_params_dsk256_lt50

# ============ STATISTICAL MATH ============

# 1. CV-RMSE (Schröder Eq. 14 — THE Li et al.-comparable metric)
cv_rmse_dsk256_lt50 = np.sqrt(np.mean(resid_dsk256_lt50**2)) / obs_dsk256_lt50.mean() * 100

# 2. Fractional RMS (per-bin normalized)
frac_rms_dsk256_lt50 = np.sqrt(np.mean((resid_dsk256_lt50 / obs_dsk256_lt50)**2)) * 100

# 3. Reduced chi-square (uses the weights from the proportional floor)
chi2_dsk256_lt50 = np.sum((resid_dsk256_lt50 * weights_dsk256_lt50)**2)
red_chi2_dsk256_lt50 = chi2_dsk256_lt50 / dof_dsk256_lt50

# 4. Mean residual (global bias check)
mean_resid_dsk256_lt50 = resid_dsk256_lt50.mean()

# 5. Residual-vs-incidence slope
inc_deg_dsk256_lt50 = np.rad2deg(geom_clean_dsk256_lt50.incidence)
slope_dsk256_lt50, intercept_dsk256_lt50 = np.polyfit(inc_deg_dsk256_lt50, obs_dsk256_lt50/pred_dsk256_lt50, 1)

# 6. Orthogonal decomposition
inc_bins_dsk256_lt50 = (inc_deg_dsk256_lt50 // 10).astype(int)
r_trend_dsk256_lt50 = np.zeros_like(resid_dsk256_lt50)
for b in np.unique(inc_bins_dsk256_lt50):
    m = inc_bins_dsk256_lt50 == b
    r_trend_dsk256_lt50[m] = resid_dsk256_lt50[m].mean()
r_other_dsk256_lt50 = resid_dsk256_lt50 - r_trend_dsk256_lt50

cv_trend_dsk256_lt50 = np.sqrt(np.mean(r_trend_dsk256_lt50**2)) / obs_dsk256_lt50.mean() * 100
cv_other_dsk256_lt50 = np.sqrt(np.mean(r_other_dsk256_lt50**2)) / obs_dsk256_lt50.mean() * 100

# ============ PRINT SUMMARY ============
print(f"{'CV-RMSE (Schröder Eq.14, vs Li et al. 5%)':45s}: {cv_rmse_dsk256_lt50:6.3f}%")
print(f"{'Fractional RMS (per-bin normalized)':45s}: {frac_rms_dsk256_lt50:6.3f}%")
print(f"{f'Reduced chi-square (dof={dof_dsk256_lt50})':45s}: {red_chi2_dsk256_lt50:6.3f}")
print(f"{'Mean residual (bias)':45s}: {mean_resid_dsk256_lt50:+.5f}")
print(f"{'Ratio-vs-incidence slope':45s}: {slope_dsk256_lt50:+.5f} /deg")
print(f"{'CV_trend (incidence systematic)':45s}: {cv_trend_dsk256_lt50:6.3f}%")
print(f"{'CV_other (within-bin / albedo)':45s}: {cv_other_dsk256_lt50:6.3f}%")
print(f"{'Quadrature check sqrt(trend²+other²)':45s}: {np.sqrt(cv_trend_dsk256_lt50**2+cv_other_dsk256_lt50**2):6.3f}%")
print("==================================================\n")

In [ ]:
# ==========================================
# 1. Setup: Calculate Base Residuals
# ==========================================
# Extract observed I/F and calculate raw residuals


observed_binned_mean_iof_lt50 = df_binned_cleaned_dsk256_lt50["mean_iof"].to_numpy()
residuals_dsk256_lt50

# Define the Coefficient of Variation (CV) lambda function

def cv_dsk256_lt50(x):
    """Coefficient of Variation (CV) = (RMS(x) / mean(observed)) * 100"""
    return np.sqrt(np.mean(x**2)) / np.mean(observed_binned_mean_iof_lt50) * 100

print(f"--- TOTAL ERROR ---")
print(f"Total Model CV ")

# Extract the DuckDB grid columns to act as our "buckets"
i_grid_dsk256_lt50 = df_binned_cleaned_dsk256_lt50["i_grid"].to_numpy()
e_grid_dsk256_lt50 = df_binned_cleaned_dsk256_lt50["e_grid"].to_numpy()
alpha_grid_dsk256_lt50 = df_binned_cleaned_dsk256_lt50["alpha_grid"].to_numpy()

# ==========================================
# 2. The 1D Decompositions (Diagnostic)
# ==========================================
def get_1d_trend(residual, grid_array):
    """Calculates the systematic error along a single geometric axis."""
    r_trend = np.zeros_like(residual)
    for b in np.unique(grid_array):
        mask = grid_array == b
        if mask.sum() > 0:
            r_trend[mask] = residual[mask].mean()
    return r_trend

# Calculate the independent 1D trends
r_trend_i_dsk256_lt50     = get_1d_trend(residuals_dsk256_lt50, i_grid_dsk256_lt50)
r_trend_e_dsk256_lt50     = get_1d_trend(residuals_dsk256_lt50, e_grid_dsk256_lt50)
r_trend_alpha_dsk256_lt50 = get_1d_trend(residuals_dsk256_lt50, alpha_grid_dsk256_lt50  )

print(f"--- 1D DIAGNOSTICS  ---")
print(f"CV_trend_incidence  : {cv_dsk256_lt50(r_trend_i_dsk256_lt50):.3f}%")
print(f"CV_trend_emission   : {cv_dsk256_lt50(r_trend_e_dsk256_lt50):.3f}%")
print(f"CV_trend_phase      : {cv_dsk256_lt50 (r_trend_alpha_dsk256_lt50):.3f}%\n")

# ==========================================
# 3. The 2D Joint Decomposition (For the Paper)
# ==========================================
# Create a blank array for the combined geometric trend
r_trend_2d_dsk256_lt50 = np.zeros_like(residuals_dsk256_lt50)

# Walk across the 2D chessboard (Incidence AND Emission)
for bi in np.unique(i_grid_dsk256_lt50):
    for be in np.unique(e_grid_dsk256_lt50):
        # Mask for pixels living in this exact (i, e) square
        mask = (i_grid_dsk256_lt50 == bi) & (e_grid_dsk256_lt50 == be)

        if mask.sum() > 0:
            r_trend_2d_dsk256_lt50[mask] = residuals_dsk256_lt50[mask].mean()

# Subtract the 2D geometric error to isolate the geological error
r_other_2d_dsk256_lt50 = residuals_dsk256_lt50 - r_trend_2d_dsk256_lt50

print("--- 2D JOINT DECOMPOSITION (The Final Result) ---")
print(f"CV_trend_2D (i,e)   : {cv_dsk256_lt50(r_trend_2d_dsk256_lt50):.3f}%  <-- (The systematic math error)")
print(f"CV_other_2D         : {cv_dsk256_lt50(r_other_2d_dsk256_lt50):.3f}%  <-- (The actual Vesta geology)")

# ==========================================
# 4. The Mathematical Proof
# ==========================================
# If the math is perfect, the dot product will be functionally zero (e.g., 1.4e-16)
dot_product = np.dot(r_trend_2d_dsk256_lt50, r_other_2d_dsk256_lt50)
print(f"\nOrthogonality check : {dot_product:.3e}")

df_binned_cleaned_dsk256_lt50["mean_residual"] = (
    df_binned_cleaned_dsk256_lt50["mean_iof"] - predicted_iof_dsk256_lt50
)


heatmap_data_dsk256_lt50 = df_binned_cleaned_dsk256_lt50.pivot_table(
    values="mean_residual", 
    index="i_grid", 
    columns="e_grid", 
    aggfunc="mean"
)

# 2. Plot the Heatmap
plt.figure(figsize=(8, 5), dpi=150)
ax = sns.heatmap(
    heatmap_data_dsk256_lt50, 
    cmap="RdBu_r",  # Red-Blue diverging: Blue = Model under-predicted, Red = Model over-predicted
    center=0,       # The center (0) is pure white
    annot=False,
    cbar_kws={'label': 'Mean Residual (Data - Model)'}
)

plt.title("Systematic Error Map: Incidence vs. Emission (lt50°)\n(Proof of Geometric Systematic Breakdown)", fontweight='bold')
plt.xlabel("Emission Angle Grid (5° Bins)")
plt.ylabel("Incidence Angle Grid (5° Bins)")
plt.show()

In [ ]:
if "Case 1 (3-Param)" in results_summary_dsk256_lt80:
    best_result_dsk256_lt80 = results_summary_dsk256_lt80["Case 1 (3-Param)"]
else:
    best_result_dsk256_lt80 = best_result_case1_dsk256_lt80.fitted_parameters

model_eval_dsk256_lt80 = HapkeModel(enable_shoe=True, enable_roughness=True, fixed_parameters={'B0': 1.03, 'h': 0.04})
model_eval_dsk256_lt80.parameters.update(best_result_dsk256_lt80)


# 2. Generate Predictions
predicted_iof_dsk256_lt80 = model_eval_dsk256_lt80._reflectance_numpy(geom_clean_dsk256_lt80)

# 3. Setup Arrays for Math
obs_dsk256_lt80  = measured_iof_dsk256_lt80
pred_dsk256_lt80 = predicted_iof_dsk256_lt80
resid_dsk256_lt80 = obs_dsk256_lt80 - pred_dsk256_lt80
n_bins_dsk256_lt80 = len(obs_dsk256_lt80)
n_params_dsk256_lt80 = len(best_result_case1_dsk256_lt80.fitted_parameters)
dof_dsk256_lt80 = n_bins_dsk256_lt80 - n_params_dsk256_lt80

# ============ STATISTICAL MATH ============

# 1. CV-RMSE (Schröder Eq. 14 — THE Li et al.-comparable metric)
cv_rmse_dsk256_lt80 = np.sqrt(np.mean(resid_dsk256_lt80**2)) / obs_dsk256_lt80.mean() * 100

# 2. Fractional RMS (per-bin normalized)
frac_rms_dsk256_lt80 = np.sqrt(np.mean((resid_dsk256_lt80 / obs_dsk256_lt80)**2)) * 100

# 3. Reduced chi-square (uses the weights from the proportional floor)
chi2_dsk256_lt80 = np.sum((resid_dsk256_lt80 * weights_dsk256_lt80)**2)
red_chi2_dsk256_lt80 = chi2_dsk256_lt80 / dof_dsk256_lt80

# 4. Mean residual (global bias check)
mean_resid_dsk256_lt80 = resid_dsk256_lt80.mean()

# 5. Residual-vs-incidence slope
inc_deg_dsk256_lt80 = np.rad2deg(geom_clean_dsk256_lt80.incidence)
slope_dsk256_lt80, intercept_dsk256_lt80 = np.polyfit(inc_deg_dsk256_lt80, obs_dsk256_lt80/pred_dsk256_lt80, 1)

# 6. Orthogonal decomposition
inc_bins_dsk256_lt80 = (inc_deg_dsk256_lt80 // 10).astype(int)
r_trend_dsk256_lt80 = np.zeros_like(resid_dsk256_lt80)
for b in np.unique(inc_bins_dsk256_lt80):
    m = inc_bins_dsk256_lt80 == b
    r_trend_dsk256_lt80[m] = resid_dsk256_lt80[m].mean()
r_other_dsk256_lt80 = resid_dsk256_lt80 - r_trend_dsk256_lt80

cv_trend_dsk256_lt80 = np.sqrt(np.mean(r_trend_dsk256_lt80**2)) / obs_dsk256_lt80.mean() * 100
cv_other_dsk256_lt80 = np.sqrt(np.mean(r_other_dsk256_lt80**2)) / obs_dsk256_lt80.mean() * 100

# ============ PRINT SUMMARY ============
print(f"{'CV-RMSE (Schröder Eq.14, vs Li et al. 5%)':45s}: {cv_rmse_dsk256_lt80:6.3f}%")
print(f"{'Fractional RMS (per-bin normalized)':45s}: {frac_rms_dsk256_lt80:6.3f}%")
print(f"{f'Reduced chi-square (dof={dof_dsk256_lt80})':45s}: {red_chi2_dsk256_lt80:6.3f}")
print(f"{'Mean residual (bias)':45s}: {mean_resid_dsk256_lt80:+.5f}")
print(f"{'Ratio-vs-incidence slope':45s}: {slope_dsk256_lt80:+.5f} /deg")
print(f"{'CV_trend (incidence systematic)':45s}: {cv_trend_dsk256_lt80:6.3f}%")
print(f"{'CV_other (within-bin / albedo)':45s}: {cv_other_dsk256_lt80:6.3f}%")
print(f"{'Quadrature check sqrt(trend²+other²)':45s}: {np.sqrt(cv_trend_dsk256_lt80**2+cv_other_dsk256_lt80**2):6.3f}%")
print("==================================================\n")

In [ ]:
# ============ DIAGNOSTIC PLOTTING ============

# 1. Update the Pandas dataframe with predictions and ratio directly
df_binned_cleaned_dsk256_lt50["modeled_iof"] = predicted_iof_dsk256_lt50
df_binned_cleaned_dsk256_lt50["ratio"] = obs_dsk256_lt50 / pred_dsk256_lt50

# 2. Since it is ALREADY a Pandas dataframe, we don't need .to_pandas()
df_plot_dsk256_lt50 = df_binned_cleaned_dsk256_lt50.copy()



# Calculate Correlation (Pearson r) between measured and modeled
corr_matrix_dsk256_lt50 = np.corrcoef(df_plot_dsk256_lt50['mean_iof'], df_plot_dsk256_lt50['modeled_iof'])
pearson_r_dsk256_lt50 = corr_matrix_dsk256_lt50[0, 1]

# Create 2x2 Grid
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle(f'Hapke Fit Diagnostics', 
             fontsize=16, fontweight='bold', y=1.02)

# ---------------------------------------------------------
# Panel [0, 0]: Modeled vs Measured I/F (Absolute Check)
# ---------------------------------------------------------
sns.scatterplot(x=df_plot_dsk256_lt50['mean_iof'], y=df_plot_dsk256_lt50['modeled_iof'], ax=axes[0, 0], alpha=0.8, color='darkorange', edgecolor=None)
axes[0, 0].set_title('Modeled vs. Measured I/F')
axes[0, 0].set_xlabel('Measured I/F')
axes[0, 0].set_ylabel('Modeled I/F')

# Plot the 1:1 Identity Line
min_val = min(df_plot_dsk256_lt50['mean_iof'].min(), df_plot_dsk256_lt50['modeled_iof'].min())
max_val = max(df_plot_dsk256_lt50['mean_iof'].max(), df_plot_dsk256_lt50['modeled_iof'].max())
axes[0, 0].plot([min_val, max_val], [min_val, max_val], 'k--', lw=2, label='1:1 Perfect Fit')

# Add Metrics Text Box
textstr = f'CV-RMSE: {cv_rmse_dsk256_lt50:.2f}%\nPearson $r$: {pearson_r_dsk256_lt50 :.4f}'
props = dict(boxstyle='round', facecolor='white', alpha=0.8, edgecolor='gray')
axes[0, 0].text(0.05, 0.95, textstr, transform=axes[0, 0].transAxes, fontsize=11,
                verticalalignment='top', bbox=props)
axes[0, 0].legend(loc='lower right')

# ---------------------------------------------------------
# Panel [0, 1]: Ratio vs Incidence
# ---------------------------------------------------------
sns.scatterplot(x=df_plot_dsk256_lt50['mean_incidence'], y=df_plot_dsk256_lt50['ratio'], ax=axes[0, 1], alpha=0.5, color='teal', edgecolor=None)
axes[0, 1].set_title('Ratio vs. Incidence')
axes[0, 1].set_xlabel('Mean Incidence (Degrees)')

# ---------------------------------------------------------
# Panel [1, 0]: Ratio vs Emission
# ---------------------------------------------------------
sns.scatterplot(x=df_plot_dsk256_lt50['mean_emission'], y=df_plot_dsk256_lt50['ratio'], ax=axes[1, 0], alpha=0.5, color='teal', edgecolor=None)
axes[1, 0].set_title('Ratio vs. Emission')
axes[1, 0].set_xlabel('Mean Emission (Degrees)')

# ---------------------------------------------------------
# Panel [1, 1]: Ratio vs Phase
# ---------------------------------------------------------
sns.scatterplot(x=df_plot_dsk256_lt50['mean_phase'], y=df_plot_dsk256_lt50 ['ratio'], ax=axes[1, 1], alpha=0.5, color='teal', edgecolor=None)
axes[1, 1].set_title('Ratio vs. Phase')
axes[1, 1].set_xlabel('Mean Phase (Degrees)')

# ---------------------------------------------------------
# Standardize Formatting
# ---------------------------------------------------------
# Apply ratio-specific formatting only to the 3 residual ratio plots
for ax in [axes[0, 1], axes[1, 0], axes[1, 1]]:
    ax.axhline(1.0, color='k', ls='--', lw=2)
    ax.set_ylim(0.7, 1.3)
    ax.set_ylabel('Measured I/F / Modeled I/F')

# Apply grid to all 4 panels
for ax in axes.flat:
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# ============ DIAGNOSTIC PLOTTING ============

# 1. Update the Pandas dataframe with predictions and ratio directly
df_binned_cleaned_dsk256_lt80["modeled_iof"] = predicted_iof_dsk256_lt80
df_binned_cleaned_dsk256_lt80["ratio"] = obs_dsk256_lt80 / pred_dsk256_lt80

# 2. Since it is ALREADY a Pandas dataframe, we don't need .to_pandas()
df_plot_dsk256_lt80 = df_binned_cleaned_dsk256_lt80.copy()



# Calculate Correlation (Pearson r) between measured and modeled
corr_matrix_dsk256_lt80 = np.corrcoef(df_plot_dsk256_lt80['mean_iof'], df_plot_dsk256_lt80['modeled_iof'])
pearson_r_dsk256_lt80 = corr_matrix_dsk256_lt80[0, 1]

# Create 2x2 Grid
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle(f'3-Parameter Hapke Fit Diagnostics | CV-RMSE: {cv_rmse_dsk256_lt80:.2f}% | $\chi^2_\\nu$: {red_chi2_dsk256_lt80:.2f}', 
             fontsize=16, fontweight='bold', y=1.02)

# ---------------------------------------------------------
# Panel [0, 0]: Modeled vs Measured I/F (Absolute Check)
# ---------------------------------------------------------
sns.scatterplot(x=df_plot_dsk256_lt80['mean_iof'], y=df_plot_dsk256_lt80['modeled_iof'], ax=axes[0, 0], alpha=0.8, color='darkorange', edgecolor=None)
axes[0, 0].set_title('Modeled vs. Measured I/F')
axes[0, 0].set_xlabel('Measured I/F')
axes[0, 0].set_ylabel('Modeled I/F')

# Plot the 1:1 Identity Line
min_val = min(df_plot_dsk256_lt80['mean_iof'].min(), df_plot_dsk256_lt80['modeled_iof'].min())
max_val = max(df_plot_dsk256_lt80['mean_iof'].max(), df_plot_dsk256_lt80['modeled_iof'].max())
axes[0, 0].plot([min_val, max_val], [min_val, max_val], 'k--', lw=2, label='1:1 Perfect Fit')

# Add Metrics Text Box
textstr = f'CV-RMSE: {cv_rmse_dsk256_lt80:.2f}%\nPearson $r$: {pearson_r_dsk256_lt80 :.4f}'
props = dict(boxstyle='round', facecolor='white', alpha=0.8, edgecolor='gray')
axes[0, 0].text(0.05, 0.95, textstr, transform=axes[0, 0].transAxes, fontsize=11,
                verticalalignment='top', bbox=props)
axes[0, 0].legend(loc='lower right')

# ---------------------------------------------------------
# Panel [0, 1]: Ratio vs Incidence
# ---------------------------------------------------------
sns.scatterplot(x=df_plot_dsk256_lt80['mean_incidence'], y=df_plot_dsk256_lt80['ratio'], ax=axes[0, 1], alpha=0.5, color='teal', edgecolor=None)
axes[0, 1].set_title('Ratio vs. Incidence')
axes[0, 1].set_xlabel('Mean Incidence (Degrees)')

# ---------------------------------------------------------
# Panel [1, 0]: Ratio vs Emission
# ---------------------------------------------------------
sns.scatterplot(x=df_plot_dsk256_lt80['mean_emission'], y=df_plot_dsk256_lt80['ratio'], ax=axes[1, 0], alpha=0.5, color='teal', edgecolor=None)
axes[1, 0].set_title('Ratio vs. Emission')
axes[1, 0].set_xlabel('Mean Emission (Degrees)')

# ---------------------------------------------------------
# Panel [1, 1]: Ratio vs Phase
# ---------------------------------------------------------
sns.scatterplot(x=df_plot_dsk256_lt80['mean_phase'], y=df_plot_dsk256_lt80 ['ratio'], ax=axes[1, 1], alpha=0.5, color='teal', edgecolor=None)
axes[1, 1].set_title('Ratio vs. Phase')
axes[1, 1].set_xlabel('Mean Phase (Degrees)')

# ---------------------------------------------------------
# Standardize Formatting
# ---------------------------------------------------------
# Apply ratio-specific formatting only to the 3 residual ratio plots
for ax in [axes[0, 1], axes[1, 0], axes[1, 1]]:
    ax.axhline(1.0, color='k', ls='--', lw=2)
    ax.set_ylim(0.7, 1.3)
    ax.set_ylabel('Measured I/F / Modeled I/F')

# Apply grid to all 4 panels
for ax in axes.flat:
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# ==========================================
# 1. EXTRACT MULTI-START DATA (Case 3 Example)
# ==========================================
# We will extract the fitted parameters from every successful multi-start
# to see how the optimizer explored the space.

successful_params_case1_dsk256_lt50 = []

# Assuming 'guesses_case3' and 'model_case3' and the 'fitter' loop are available.
# (If you didn't save all results previously, we just quickly re-run the loop to capture them)
for guess in initial_guesses_case1_dsk256_lt50:
    model_case1_dsk256_lt50.parameters.update(guess)
    res_dsk256_lt50 = fitter.fit(model_case1_dsk256_lt50 , geom_clean_dsk256_lt50, measured_iof_dsk256_lt50, weights_dsk256_lt50)

    if res_dsk256_lt50.metadata["success"]:
        # Save the cost and the fitted parameters
        row = {'Cost': res_dsk256_lt50.objective_value}
        row.update(res_dsk256_lt50.fitted_parameters)
        successful_params_case1_dsk256_lt50.append(row)
        # keep legacy variable name used later
        successful_params_case1_dsk256_lt50 = successful_params_case1_dsk256_lt50

df_params_clean_dsk256_lt50 = pd.DataFrame(successful_params_case1_dsk256_lt50)

# Filter out completely failed runaway fits (keep costs reasonably close to the minimum)
min_cost_dsk256_lt50 = df_params_clean_dsk256_lt50['Cost'].min()
df_params_clean_dsk256_lt50 = df_params_clean_dsk256_lt50[df_params_clean_dsk256_lt50['Cost'] < min_cost_dsk256_lt50 * 1.5].drop(columns=['Cost'])

print(f"Plotting landscape from {len(df_params_clean_dsk256_lt50)} successful multi-start convergences.")

# ==========================================

# 2. PLOT: CORRELATION HEATMAP
# ==========================================
plt.figure(figsize=(8, 6))
plt.title("Case 1: Parameter Correlation Heatmap for i,e <50", fontsize=14, fontweight='bold', pad=20)

# Calculate the correlation matrix
corr_matrix = df_params_clean_dsk256_lt50.corr()

# Create a custom diverging colormap (Blue for negative, Red for positive)
cmap = sns.diverging_palette(230, 20, as_cmap=True)

# Plot the heatmap
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap=cmap, vmin=-1, vmax=1, 
            square=True, linewidths=.5, cbar_kws={"shrink": .8})
plt.tight_layout()
plt.show()

In [ ]:
sns.set_theme(style="ticks", font_scale=1.1)

# 2. Create the PairGrid (using 'height' to make the panels larger)
pairgrid_dsk256_lt50 = sns.PairGrid(df_params_clean_dsk256_lt50, diag_sharey=False, corner=True, height=3.5)

# 3. Map the lower triangle: Smooth KDE contours + precise scatter points
pairgrid_dsk256_lt50.map_lower(sns.kdeplot, fill=True, cmap="mako", alpha=0.5, levels=6)
pairgrid_dsk256_lt50.map_lower(sns.scatterplot, color=".1", s=25, alpha=0.7, edgecolor="white", linewidth=0.5)

# 4. Map the diagonal: Clean histograms showing the tight convergence
pairgrid_dsk256_lt50.map_diag(sns.histplot, color="#2c7fb8", kde=True, linewidth=1, alpha=0.6)

# 5. THE CRITICAL FIX: Disable the confusing 1e-6 scientific offset
for ax in pairgrid_dsk256_lt50.axes.flatten():
    if ax is not None:
        # Force plain text formatting for the tick labels
        ax.ticklabel_format(useOffset=False, style='plain', axis='both')
        # Rotate x-axis labels slightly so they don't overlap
        ax.tick_params(axis='x', rotation=45)

# 6. Set proper parameter labels (using LaTeX for the axis titles)
pairgrid_dsk256_lt50.axes[1, 0].set_ylabel('Phase Function (g)')
pairgrid_dsk256_lt50.axes[2, 0].set_ylabel('Roughness ($\\bar{\\theta}$)')
pairgrid_dsk256_lt50.axes[2, 0].set_xlabel('Albedo (w)')
pairgrid_dsk256_lt50.axes[2, 1].set_xlabel('Phase Function (g)')
pairgrid_dsk256_lt50.axes[2, 2].set_xlabel('Roughness ($\\bar{\\theta}$)')

# 7. Add the master title
pairgrid_dsk256_lt50.figure.suptitle("Case 1: Multi-Start Convergence Landscape ($<50^\circ$ Incidence)", 
                  fontsize=18, fontweight='bold', y=1.05)

plt.tight_layout()
plt.show()

**Note:** the lt80 pair-grid below references `df_params_clean_dsk256_lt80`, which was never defined in the recovered session — needs an lt80 multi-start extraction cell mirroring cell 24 before this will run. Not fabricated here to keep this reconstruction faithful to what was actually executed on 2026-08-12.

In [ ]:
############################# i,e < 80 degrees #############################



pairgrid_dsk256_lt80 = sns.PairGrid(df_params_clean_dsk256_lt80, diag_sharey=False, corner=True, height=3.5)

# 3. Map the lower triangle: Smooth KDE contours + precise scatter points
pairgrid_dsk256_lt80.map_lower(sns.kdeplot, fill=True, cmap="mako", alpha=0.5, levels=6)
pairgrid_dsk256_lt80.map_lower(sns.scatterplot, color=".1", s=25, alpha=0.7, edgecolor="white", linewidth=0.5)

# 4. Map the diagonal: Clean histograms showing the tight convergence
pairgrid_dsk256_lt80.map_diag(sns.histplot, color="#2c7fb8", kde=True, linewidth=1, alpha=0.6)

# 5. THE CRITICAL FIX: Disable the confusing 1e-6 scientific offset
for ax in pairgrid_dsk256_lt80.axes.flatten():
    if ax is not None:
        # Force plain text formatting for the tick labels
        ax.ticklabel_format(useOffset=False, style='plain', axis='both')
        # Rotate x-axis labels slightly so they don't overlap
        ax.tick_params(axis='x', rotation=45)

# 6. Set proper parameter labels (using LaTeX for the axis titles)
pairgrid_dsk256_lt80.axes[1, 0].set_ylabel('Phase Function (g)')
pairgrid_dsk256_lt80.axes[2, 0].set_ylabel('Roughness ($\\bar{\\theta}$)')
pairgrid_dsk256_lt80.axes[2, 0].set_xlabel('Albedo (w)')
pairgrid_dsk256_lt80.axes[2, 1].set_xlabel('Phase Function (g)')
pairgrid_dsk256_lt80.axes[2, 2].set_xlabel('Roughness ($\\bar{\\theta}$)')

# 7. Add the master title
pairgrid_dsk256_lt80.figure.suptitle("Case 1: Multi-Start Convergence Landscape ($<80^\circ$ Incidence)", 
                  fontsize=18, fontweight='bold', y=1.05)

plt.tight_layout()
plt.show()